In [1]:
!pip install dtaidistance
!pip install fastdtw

In [2]:
import json
import glob
import os
import random
import math
import numpy as np
import pandas as pd
import pywt
from scipy.interpolate import CubicSpline
from scipy.signal import butter, filtfilt, resample_poly

from scipy.spatial.distance import euclidean, cdist

# Deep Learning Framework
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset

# Machine Learning Ensemble Classifiers & Metrics
from catboost import CatBoostClassifier, Pool
import xgboost as xgb
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score, precision_recall_fscore_support,
    matthews_corrcoef, roc_auc_score, average_precision_score, log_loss, brier_score_loss,
    confusion_matrix, precision_recall_curve, auc
)
from sklearn.linear_model import LogisticRegression
from sklearn.utils.class_weight import compute_class_weight
from scipy.signal import hilbert


import keras
if not hasattr(keras.layers, 'normalization'):
    class LegacyNormModule:
        BatchNormalization = keras.layers.BatchNormalization
    keras.layers.normalization = LegacyNormModule()

# ============================================================================
# DETERMINISTIC SEED & DEPLOYMENT TARGET STRATEGY
# ============================================================================
def set_experimental_reproducibility(seed=42):
    """
    Fixes pseudo-random number generator states across all environments
    to enforce strict cross-validation integrity and model stability.
    """
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.cuda.set_device(0) 
        
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Initialize the global experimental parameters
GLOBAL_SEED = 42
set_experimental_reproducibility(GLOBAL_SEED)

# Runtime Execution Device Mapping Target
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Runtime execution backend designated to: {DEVICE}")

def seed_worker(worker_id):
    """
    Worker initialization function for PyTorch DataLoaders to eliminate
    asynchronous data sampling leakage across threads.
    """
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

Runtime execution backend designated to: cuda


In [3]:


# Path to the JSON file
file_path = "/kaggle/input/datasets/abdallasadik/pads-pd-dataset/pads-parkinsons-disease-smartwatch-dataset-1.0.0/patients/patient_001.json"

# Open and load the JSON file
with open(file_path, "r") as f:
    data = json.load(f)

# Print the contents
print(data)

{'resource_type': 'patient', 'id': '001', 'study_id': 'PADS', 'condition': 'Healthy', 'disease_comment': '-', 'age_at_diagnosis': 56, 'age': 56, 'height': 173, 'weight': 78, 'gender': 'male', 'handedness': 'right', 'appearance_in_kinship': True, 'appearance_in_first_grade_kinship': True, 'effect_of_alcohol_on_tremor': 'Unknown'}


In [4]:
ORIGINAL_FS = 100
TARGET_FS = 64
WINDOW_SIZE = 256

In [5]:


# Path to the patients folder
patients_dir = "/kaggle/input/datasets/abdallasadik/pads-pd-dataset/pads-parkinsons-disease-smartwatch-dataset-1.0.0/patients"

# List to store extracted data
records = []

# Iterate through all JSON files
for filename in sorted(os.listdir(patients_dir)):
    if filename.endswith(".json"):
        file_path = os.path.join(patients_dir, filename)

        with open(file_path, "r") as f:
            data = json.load(f)

        records.append({
            "id": data.get("id"),
            "condition": data.get("condition")
        })

# Create DataFrame
df = pd.DataFrame(records)

# Display the table
print(df)

      id                 condition
0    001                   Healthy
1    002  Other Movement Disorders
2    003                   Healthy
3    004               Parkinson's
4    005               Parkinson's
..   ...                       ...
464  465               Parkinson's
465  466                   Healthy
466  467               Parkinson's
467  468               Parkinson's
468  469               Parkinson's

[469 rows x 2 columns]


In [6]:
# Frequency of unique values in the last column
frequency = df.iloc[:, -1].value_counts()

print(frequency)

condition
Parkinson's                 276
Healthy                      79
Other Movement Disorders     60
Essential Tremor             28
Atypical Parkinsonism        15
Multiple Sclerosis           11
Name: count, dtype: int64


In [7]:
# Remove all rows where the second column (index 1) is 'Healthy'
second_col_name = df.columns[1]
df = df[df[second_col_name] != "Healthy"]

# Check the new frequencies of the target/condition column (last column)
print(df.iloc[:, -1].value_counts())

condition
Parkinson's                 276
Other Movement Disorders     60
Essential Tremor             28
Atypical Parkinsonism        15
Multiple Sclerosis           11
Name: count, dtype: int64


In [8]:
# Rename 'Parkinson's' (first) and 'Atypical Parkinsonism' (second last) to 'PD'
df.iloc[:, -1] = df.iloc[:, -1].replace({
    "Parkinson's": "PD",
    "Atypical Parkinsonism": "PD"
})

# Check the new frequencies
print(df.iloc[:, -1].value_counts())

condition
PD                          291
Other Movement Disorders     60
Essential Tremor             28
Multiple Sclerosis           11
Name: count, dtype: int64


In [9]:
df = df[df.iloc[:, -1] != 'Healthy'].copy()

In [10]:
print(df.shape)

(390, 2)


In [11]:
import json

# Define the absolute file path to the target patient metadata record
json_path = "/kaggle/input/datasets/abdallasadik/pads-pd-dataset/pads-parkinsons-disease-smartwatch-dataset-1.0.0/patients/patient_001.json"

try:
    with open(json_path, "r", encoding="utf-8") as f:  # Fixed mode from 'file_r' to 'r'
        patient_metadata = json.load(f)
        
    # Pretty-print the entire nested dictionary structure
    print(json.dumps(patient_metadata, indent=4))

except FileNotFoundError:
    print(f"Error: The file at {json_path} could not be located. Double-check your directory tree structure.")
except json.JSONDecodeError:
    print("Error: Failed to parse the file. The file format might be corrupted or incomplete.")

{
    "resource_type": "patient",
    "id": "001",
    "study_id": "PADS",
    "condition": "Healthy",
    "disease_comment": "-",
    "age_at_diagnosis": 56,
    "age": 56,
    "height": 173,
    "weight": 78,
    "gender": "male",
    "handedness": "right",
    "appearance_in_kinship": true,
    "appearance_in_first_grade_kinship": true,
    "effect_of_alcohol_on_tremor": "Unknown"
}


In [12]:


# Path to the patients folder
patients_dir = "/kaggle/input/datasets/abdallasadik/pads-pd-dataset/pads-parkinsons-disease-smartwatch-dataset-1.0.0/patients"

# List to store extracted data
records = []

# Iterate through all JSON files
for filename in sorted(os.listdir(patients_dir)):
    if filename.endswith(".json"):
        file_path = os.path.join(patients_dir, filename)

        with open(file_path, "r") as f:
            data = json.load(f)

        records.append({
            "id": data.get("id"),
            "condition": data.get("condition")
        })

# Create DataFrame
df = pd.DataFrame(records)

# Display the table
print(df)


      id                 condition
0    001                   Healthy
1    002  Other Movement Disorders
2    003                   Healthy
3    004               Parkinson's
4    005               Parkinson's
..   ...                       ...
464  465               Parkinson's
465  466                   Healthy
466  467               Parkinson's
467  468               Parkinson's
468  469               Parkinson's

[469 rows x 2 columns]


In [13]:

# 1. Update pattern with a wildcard '*' to catch all 469 response files dynamically
file_pattern = "/kaggle/input/datasets/abdallasadik/pads-pd-dataset/pads-parkinsons-disease-smartwatch-dataset-1.0.0/questionnaire/questionnaire_response_*.json"
all_files = sorted(glob.glob(file_pattern))

all_patient_records = []

# 2. Iterate through each patient profile and safely parse the content
for file_path in all_files:
    try:
        with open(file_path, "r", encoding="utf-8") as f:
            data = json.load(f)
            
        # --- FIXED INDENTATION: Safely inside the try block now ---
        patient_id = data.get("subject_id", "Unknown")
        
        # Format the patient index name cleanly
        patient_row = {"patient_number": f"patient_{patient_id}"}
        
        # Extract individual answers from the nested item matrix
        if "item" in data:
            for question in data["item"]:
                col_name = f"q{int(question['link_id'])}"
                
                # Convert boolean True/False to clean numeric 1/0 flags
                patient_row[col_name] = 1 if question["answer"] is True else 0
                
        all_patient_records.append(patient_row)
        
    except Exception as e:
        print(f"Skipping corrupt or missing file {file_path}: {e}")

# 3. Transform into our unified, flat Pandas DataFrame
df1 = pd.DataFrame(all_patient_records)

# Handle empty-check safeguard just in case paths are shifted on Kaggle
if not df.empty:
    df1.set_index("patient_number", inplace=True)
    # Sort columns chronologically from q1 to q30
    question_cols = sorted(df1.columns, key=lambda x: int(x[1:]))
    df1 = df1[question_cols]

    print(f"Successfully processed {df.shape[0]} patients across {df.shape[1]} non-motor symptoms.")
else:
    print("No records found. Verify that the questionnaire path matches your Kaggle input directory layout.")

# Verify the multi-row table structure
df1.head()

Successfully processed 469 patients across 2 non-motor symptoms.


,q1,q2,q3,q4,q5,q6,q7,q8,q9,q10,...,q21,q22,q23,q24,q25,q26,q27,q28,q29,q30
patient_number,,,,,,,,,,,,,,,,,,,,,
patient_001,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
patient_002,1,1,0,0,1,0,0,0,1,0,...,1,1,1,0,1,0,1,0,1,0
patient_003,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
patient_004,0,1,0,1,0,0,0,1,1,1,...,1,1,1,0,0,1,1,1,0,0
patient_005,1,1,1,0,0,0,0,1,0,0,...,0,1,1,1,1,1,1,0,0,0


In [14]:
# Extract the last column from df and copy the raw array straight into df1
df1['disease'] = df.iloc[:, -1].to_numpy()

# Verify the first 10 rows to ensure values are populated properly
print(df1.head(10))

                q1  q2  q3  q4  q5  q6  q7  q8  q9  q10  ...  q22  q23  q24  \
patient_number                                           ...                  
patient_001      0   0   0   0   0   0   0   0   0    0  ...    0    0    0   
patient_002      1   1   0   0   1   0   0   0   1    0  ...    1    1    0   
patient_003      0   0   0   0   0   0   0   0   0    0  ...    0    0    0   
patient_004      0   1   0   1   0   0   0   1   1    1  ...    1    1    0   
patient_005      1   1   1   0   0   0   0   1   0    0  ...    1    1    1   
patient_006      0   1   0   1   1   0   0   1   1    0  ...    0    1    0   
patient_007      1   0   1   0   1   1   1   1   1    0  ...    0    1    0   
patient_008      0   1   1   1   1   0   0   0   1    0  ...    0    0    0   
patient_009      0   1   0   0   0   0   0   0   0    0  ...    0    1    0   
patient_010      0   0   0   0   0   0   0   1   1    0  ...    0    1    0   

                q25  q26  q27  q28  q29  q30       

In [15]:
# 1. Remove rows where 'Healthy' is present
df_pd_dd = df1[df1['disease'] != 'Healthy'].copy()

# 3. Map 'PD' to 0 and 'DD' to 1
# 4. Print out class counts to verify the mapping worked perfectly
print("Encoded PD vs. DD Class Distribution:")
print(df_pd_dd['disease'].value_counts())

print("\nFirst 10 rows of the encoded PD/DD DataFrame:")
print(df_pd_dd.head(10))

Encoded PD vs. DD Class Distribution:
disease
Parkinson's                 276
Other Movement Disorders     60
Essential Tremor             28
Atypical Parkinsonism        15
Multiple Sclerosis           11
Name: count, dtype: int64

First 10 rows of the encoded PD/DD DataFrame:
                q1  q2  q3  q4  q5  q6  q7  q8  q9  q10  ...  q22  q23  q24  \
patient_number                                           ...                  
patient_002      1   1   0   0   1   0   0   0   1    0  ...    1    1    0   
patient_004      0   1   0   1   0   0   0   1   1    1  ...    1    1    0   
patient_005      1   1   1   0   0   0   0   1   0    0  ...    1    1    1   
patient_006      0   1   0   1   1   0   0   1   1    0  ...    0    1    0   
patient_007      1   0   1   0   1   1   1   1   1    0  ...    0    1    0   
patient_008      0   1   1   1   1   0   0   0   1    0  ...    0    0    0   
patient_009      0   1   0   0   0   0   0   0   0    0  ...    0    1    0   
patient_01

In [16]:
df = df[df.iloc[:, -1] != 'Healthy']

In [17]:
# Rename 'Parkinson's' and 'Atypical Parkinsonism' to 'PD' and reflect it in the dataframe
last_col = df.columns[-1]
df[last_col] = df[last_col].replace({
    "Parkinson's": "PD",
    "Atypical Parkinsonism": "PD"
})

# Check the new frequencies
print(df[last_col].value_counts())

condition
PD                          291
Other Movement Disorders     60
Essential Tremor             28
Multiple Sclerosis           11
Name: count, dtype: int64


In [18]:
print(df.shape)

(390, 2)


In [19]:
p_ids = df.iloc[:,0].values
condition_ids = df.iloc[:,1].values
print(condition_ids[0:5])

['Other Movement Disorders' 'PD' 'PD' 'PD' 'Other Movement Disorders']


In [20]:
# Rename conditions
df_pd_dd.iloc[:, -1] = df_pd_dd.iloc[:, -1].replace({
    "Parkinson's": "PD",
    "Atypical Parkinsonism": "PD",
    "Healthy": "HC"
})

# Rename all remaining categories as DD
df_pd_dd.iloc[:, -1] = df_pd_dd.iloc[:, -1].where(
    df_pd_dd.iloc[:, -1].isin(["PD", "HC"]),
    "DD"
)

# Check the new frequencies
print(df_pd_dd.iloc[:, -1].value_counts())

disease
PD    291
DD     99
Name: count, dtype: int64


In [21]:
print(len(condition_ids))

390


In [22]:

# 1. Update pattern with a wildcard '*' to catch all 469 response files dynamically
file_pattern = "/kaggle/input/datasets/abdallasadik/pads-pd-dataset/pads-parkinsons-disease-smartwatch-dataset-1.0.0/questionnaire/questionnaire_response_*.json"
all_files = sorted(glob.glob(file_pattern))

all_patient_records = []

# 2. Iterate through each patient profile and safely parse the content
for file_path in all_files:
    try:
        with open(file_path, "r", encoding="utf-8") as f:
            data = json.load(f)
            
        # --- FIXED INDENTATION: Safely inside the try block now ---
        patient_id = data.get("subject_id", "Unknown")
        
        # Format the patient index name cleanly
        patient_row = {"patient_number": f"patient_{patient_id}"}
        
        # Extract individual answers from the nested item matrix
        if "item" in data:
            for question in data["item"]:
                col_name = f"q{int(question['link_id'])}"
                
                # Convert boolean True/False to clean numeric 1/0 flags
                patient_row[col_name] = 1 if question["answer"] is True else 0
                
        all_patient_records.append(patient_row)
        
    except Exception as e:
        print(f"Skipping corrupt or missing file {file_path}: {e}")

# 3. Transform into our unified, flat Pandas DataFrame
df1 = pd.DataFrame(all_patient_records)

# Handle empty-check safeguard just in case paths are shifted on Kaggle
if not df.empty:
    df1.set_index("patient_number", inplace=True)
    # Sort columns chronologically from q1 to q30
    question_cols = sorted(df1.columns, key=lambda x: int(x[1:]))
    df1 = df1[question_cols]

    print(f"Successfully processed {df.shape[0]} patients across {df.shape[1]} non-motor symptoms.")
else:
    print("No records found. Verify that the questionnaire path matches your Kaggle input directory layout.")

# Verify the multi-row table structure
df1.head()

Successfully processed 390 patients across 2 non-motor symptoms.


,q1,q2,q3,q4,q5,q6,q7,q8,q9,q10,...,q21,q22,q23,q24,q25,q26,q27,q28,q29,q30
patient_number,,,,,,,,,,,,,,,,,,,,,
patient_001,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
patient_002,1,1,0,0,1,0,0,0,1,0,...,1,1,1,0,1,0,1,0,1,0
patient_003,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
patient_004,0,1,0,1,0,0,0,1,1,1,...,1,1,1,0,0,1,1,1,0,0
patient_005,1,1,1,0,0,0,0,1,0,0,...,0,1,1,1,1,1,1,0,0,0


In [23]:
print(df_pd_dd.head(10))

                q1  q2  q3  q4  q5  q6  q7  q8  q9  q10  ...  q22  q23  q24  \
patient_number                                           ...                  
patient_002      1   1   0   0   1   0   0   0   1    0  ...    1    1    0   
patient_004      0   1   0   1   0   0   0   1   1    1  ...    1    1    0   
patient_005      1   1   1   0   0   0   0   1   0    0  ...    1    1    1   
patient_006      0   1   0   1   1   0   0   1   1    0  ...    0    1    0   
patient_007      1   0   1   0   1   1   1   1   1    0  ...    0    1    0   
patient_008      0   1   1   1   1   0   0   0   1    0  ...    0    0    0   
patient_009      0   1   0   0   0   0   0   0   0    0  ...    0    1    0   
patient_010      0   0   0   0   0   0   0   1   1    0  ...    0    1    0   
patient_011      0   0   0   0   0   0   0   0   0    0  ...    1    0    0   
patient_012      0   0   0   0   0   0   0   0   0    0  ...    0    1    0   

                q25  q26  q27  q28  q29  q30  disea

In [24]:
# 1. Filter df1 to keep only 'PD' and 'DD' rows (using .copy() to avoid warnings)
df_pd_dd = df_pd_dd[df_pd_dd['disease'].isin(['PD', 'DD'])].copy()

# 2. Map 'PD' to 0 and 'DD' to 1
df_pd_dd['disease'] = df_pd_dd['disease'].map({'PD': 0, 'DD': 1})

# 3. Print out class counts to verify the mapping worked perfectly
print("Encoded PD vs. DD Class Distribution:")
print(df_pd_dd['disease'].value_counts())

print("\nFirst 10 rows of the encoded PD/DD DataFrame:")
print(df_pd_dd.head(10))

Encoded PD vs. DD Class Distribution:
disease
0    291
1     99
Name: count, dtype: int64

First 10 rows of the encoded PD/DD DataFrame:
                q1  q2  q3  q4  q5  q6  q7  q8  q9  q10  ...  q22  q23  q24  \
patient_number                                           ...                  
patient_002      1   1   0   0   1   0   0   0   1    0  ...    1    1    0   
patient_004      0   1   0   1   0   0   0   1   1    1  ...    1    1    0   
patient_005      1   1   1   0   0   0   0   1   0    0  ...    1    1    1   
patient_006      0   1   0   1   1   0   0   1   1    0  ...    0    1    0   
patient_007      1   0   1   0   1   1   1   1   1    0  ...    0    1    0   
patient_008      0   1   1   1   1   0   0   0   1    0  ...    0    0    0   
patient_009      0   1   0   0   0   0   0   0   0    0  ...    0    1    0   
patient_010      0   0   0   0   0   0   0   1   1    0  ...    0    1    0   
patient_011      0   0   0   0   0   0   0   0   0    0  ...    1    0   

In [25]:
dataframe = df_pd_dd
target_col='disease'

X_pd_dd = dataframe.drop(columns=[target_col]).copy()
y_pd_dd = dataframe[target_col].to_numpy()

In [26]:
BENCHMARK_TASKS = [
    "Relaxed", "RelaxedTask",  
    "StretchHold", "HoldWeight", "DrinkGlas", "CrossArms", 
    "TouchNose", "Entrainment"
]

In [27]:
SIMULTANEOUS_TASKS = {
    "Relaxed", "RelaxedTask", 
    "CrossArms", "Entrainment"
}

In [28]:
SEQUENTIAL_TASKS = {
    "StretchHold", "HoldWeight", "DrinkGlas", "TouchNose"
}


In [29]:

def align_unaligned(L_signal, R_signal):
    """Method 1: Unaligned baseline (Direct sample comparison without correction)."""
    return L_signal, R_signal, 1.0

In [30]:
import numpy as np
from dtaidistance import dtw

def align_dynamic_time_warp(L_signal, R_signal):
    """Method 2: Maximum-speed C-accelerated DTW with window constraints."""
    l_1d = np.linalg.norm(L_signal, axis=0)
    r_1d = np.linalg.norm(R_signal, axis=0)
    
    # Use a window constraint (e.g., 5% to 10% of signal length) to restrict 
    # the search space and skip heavy O(N^2) grid calculations.
    window_size = int(max(len(l_1d), len(r_1d)) * 0.05)
    
    # Use use_c=True to force the C-backend for near-instantaneous execution
    path = dtw.warping_path(l_1d, r_1d, window=window_size, use_c=True)
    
    path_a_idx = [p[0] for p in path]
    path_b_idx = [p[1] for p in path]
    
    aligned_a = L_signal[:, path_a_idx]
    aligned_b = R_signal[:, path_b_idx]
    
    return aligned_a, aligned_b, 0.85

In [31]:
def align_learned_monotone(L_signal, R_signal):
    """Method 3: Learned Monotone Alignment (Placeholder for data-driven parametric mapping)."""
    # In a full deep learning pipeline, this would pass through a trained monotonic network.
    # Here, we simulate a linear/monotone interpolation constraint matching temporal lengths smoothly.
    target_len = min(L_signal.shape[1], R_signal.shape[1])
    indices_l = np.linspace(0, L_signal.shape[1] - 1, target_len).astype(int)
    indices_r = np.linspace(0, R_signal.shape[1] - 1, target_len).astype(int)
    
    aligned_a = L_signal[:, indices_l]
    aligned_b = R_signal[:, indices_r]
    return aligned_a, aligned_b, 0.90 # Simulated model confidence score

In [32]:
def align_task_marker(L_signal, R_signal, task_name):
    """Method 4: Task-Marker Alignment (Protocol-driven structural segmentation)."""
    total_len_l = L_signal.shape[1]
    total_len_r = R_signal.shape[1]
    
    # Split into default protocol halves (Left-hand execution vs Right-hand execution blocks)
    left_segment = L_signal[:, :total_len_l // 2]
    right_segment = R_signal[:, total_len_r // 2:]
    
    # Compute energy distribution to retain a confidence score for phase alignment
    energy_l = np.sum(np.abs(left_segment))
    energy_r = np.sum(np.abs(right_segment))
    confidence = float(np.clip((min(energy_l, energy_r) / (max(energy_l, energy_r) + 1e-6)) * 1.2, 0.0, 1.0))
    
    return left_segment, right_segment, confidence

In [33]:

def load_patient_timeseries(patient_id, base_dir, sensor_mode="accelerometer", alignment_method="task-marker"):
    
    patient_str = f"{int(patient_id):03d}"
    ts_dir = os.path.join(base_dir, "movement", "timeseries")
    processed_tasks = {}

    if sensor_mode.lower() == "accelerometer":
        start_idx, end_idx = 0, 3
        b, a = butter(5, [0.1, 20], btype="bandpass", fs=TARGET_FS)
    elif sensor_mode.lower() == "gyroscope":
        start_idx, end_idx = 3, 6
        b, a = butter(5, [0.05, 20], btype="bandpass", fs=TARGET_FS)
    else:
        raise ValueError("sensor_mode must be 'accelerometer' or 'gyroscope'")

    for task in BENCHMARK_TASKS:
        left_file = os.path.join(ts_dir, f"{patient_str}_{task}_LeftWrist.txt")
        right_file = os.path.join(ts_dir, f"{patient_str}_{task}_RightWrist.txt")

        if not (os.path.exists(left_file) and os.path.exists(right_file)):
            continue

        try:
            left_raw = np.loadtxt(left_file, delimiter=",")[:, 1:].T   
            right_raw = np.loadtxt(right_file, delimiter=",")[:, 1:].T  
        except Exception:
            continue  

        left_axes = left_raw[start_idx:end_idx, :]   
        right_axes = right_raw[start_idx:end_idx, :]  

        recordings = {}
        if left_axes.shape[1] == 2048 and right_axes.shape[1] == 2048:
            recordings[f"{task}_1"] = (left_axes[:, :1024], right_axes[:, :1024])
            recordings[f"{task}_2"] = (left_axes[:, 1024:], right_axes[:, 1024:])
        else:
            recordings[task] = (left_axes, right_axes)

        for name, (L_signals, R_signals) in recordings.items():
            if L_signals.shape[1] <= 50:
                continue
            L_signals = L_signals[:, 50:]
            R_signals = R_signals[:, 50:]

            L_ds = resample_poly(L_signals, up=TARGET_FS, down=ORIGINAL_FS, axis=1)
            R_ds = resample_poly(R_signals, up=TARGET_FS, down=ORIGINAL_FS, axis=1)

            L_filt = filtfilt(b, a, L_ds, axis=1, padtype='odd', padlen=3*5)
            R_filt = filtfilt(b, a, R_ds, axis=1, padtype='odd', padlen=3*5)

            # Resolve base task name
            base_task_name = name
            for suffix in ["_1", "_2"]:
                if base_task_name.endswith(suffix):
                    base_task_name = base_task_name[:-len(suffix)]

            # --- SELECTABLE BENCHMARK ALIGNMENT ROUTE ---
            if base_task_name in SEQUENTIAL_TASKS:
                if alignment_method == "unaligned":
                    L_filt, R_filt, conf = align_unaligned(L_filt, R_filt)
                elif alignment_method == "dynamic-time-warping":
                    L_filt, R_filt, conf = align_dynamic_time_warp(L_filt, R_filt)
                elif alignment_method == "learned-monotone":
                    L_filt, R_filt, conf = align_learned_monotone(L_filt, R_filt)
                elif alignment_method == "task-marker":
                    L_filt, R_filt, conf = align_task_marker(L_filt, R_filt, name)
                else:
                    raise ValueError(f"Unknown alignment method: {alignment_method}")
            elif base_task_name in SIMULTANEOUS_TASKS:
                # Simultaneous tasks stay parallel naturally
                pass
            # ---------------------------------------------

            min_len = min(L_filt.shape[1], R_filt.shape[1])
            L_filt = L_filt[:, :min_len]
            R_filt = R_filt[:, :min_len]

            # Store full multi-axis arrays (shape: [axes, time])
            processed_tasks[name] = {
                "left": L_filt.astype(np.float32),
                "right": R_filt.astype(np.float32)
            }

    return processed_tasks


def dhwt_segment(rec_acc, rec_gyro, patient_class):
    """Applies Differential Hopping Windowing Technique and returns a structured dictionary per task."""
    patient_class = patient_class.upper()
    overlap_mapping = {"PD": 0, "HC": 0, "DD": 0}
    if patient_class not in overlap_mapping:
        raise ValueError("patient_class must be 'PD', 'HC', or 'DD'")
        
    overlap = overlap_mapping[patient_class]
    hop = int(WINDOW_SIZE * (1 - overlap))
    
    all_tasks = set(rec_acc.keys()).union(set(rec_gyro.keys()))
    segmented_tasks = {}

    for task_name in sorted(all_tasks):
        acc_data = rec_acc.get(task_name, None)
        gyro_data = rec_gyro.get(task_name, None)
        
        # Determine total length from whichever modality is present
        sample_modality = acc_data if acc_data is not None else gyro_data
        total_len = sample_modality["left"].shape[1]
        
        if total_len < WINDOW_SIZE:
            continue

        task_windows = {"left_acc": [], "right_acc": [], "left_gyro": [], "right_gyro": []}

        for start in range(0, total_len - WINDOW_SIZE + 1, hop):
            end = start + WINDOW_SIZE
            
            if acc_data is not None:
                task_windows["left_acc"].append(acc_data["left"][:, start:end].T)
                task_windows["right_acc"].append(acc_data["right"][:, start:end].T)
            
            if gyro_data is not None:
                task_windows["left_gyro"].append(gyro_data["left"][:, start:end].T)
                task_windows["right_gyro"].append(gyro_data["right"][:, start:end].T)

        # Convert lists to numpy arrays if populated
        segmented_tasks[task_name] = {
            k: np.stack(v, axis=0) if len(v) > 0 else np.empty((0, WINDOW_SIZE, 3), dtype=np.float32)
            for k, v in task_windows.items()
        }
        
    return segmented_tasks


# ============================================================
# Main Execution Entry Point
# ============================================================
def prepare_patient_data(patient_id, patient_class,
                         base_dir="/kaggle/input/datasets/abdallasadik/pads-pd-dataset/pads-parkinsons-disease-smartwatch-dataset-1.0.0",
                         alignment_method="task-marker"):
   
    rec_acc = load_patient_timeseries(patient_id, base_dir, sensor_mode="accelerometer", alignment_method=alignment_method)
    rec_gyro = load_patient_timeseries(patient_id, base_dir, sensor_mode="gyroscope", alignment_method=alignment_method)
    
    return rec_acc,rec_gyro 

In [34]:
import torch
import torch.nn as nn
import numpy as np

def build_gait_cnn_model(X_sample, num_layers=2, learning_rate=0.001, batch_size=32):
    """
    Dynamically builds a 2D CNN model enforcing 4D tensors [N, C, H, W] (or [N, C, T, 1]) 
    by padding single dimensions with a trailing 1 as requested.
    """
    class DynamicGaitCNN(nn.Module):
        def __init__(self, input_shape, num_layers):
            super().__init__()
            
            # Ensure effective_shape is strictly 3D: (C, H, W) -> resulting in 4D batch tensors [N, C, H, W]
            if len(input_shape) == 1:
                # e.g., (T,) -> (1, T, 1)
                channels, h, w = 1, input_shape[0], 1
                self.transpose_needed = False
            elif len(input_shape) == 2:
                # e.g., (C, T) or (T, C)
                if input_shape[0] > input_shape[1]:
                    self.transpose_needed = True
                    channels, h, w = input_shape[1], input_shape[0], 1
                else:
                    self.transpose_needed = False
                    channels, h, w = input_shape[0], input_shape[1], 1
            elif len(input_shape) == 3:
                channels, h, w = input_shape[0], input_shape[1], input_shape[2]
                self.transpose_needed = False
            else:
                channels, h, w = input_shape[0], input_shape[1], 1
                self.transpose_needed = False

            effective_shape = (channels, h, w)
            in_channels = channels
            layers = []
            
            for i in range(num_layers):
                out_channels = min(32 * (2 ** i), 128)
                # Use standard 2D convolutions with padding safe for width=1 if w=1
                pad_w = 0 if w == 1 else 1
                kernel_w = 1 if w == 1 else 3
                
                layers.append(nn.Conv2d(in_channels, out_channels, kernel_size=(3, kernel_w), padding=(1, pad_w)))
                layers.append(nn.BatchNorm2d(out_channels))
                layers.append(nn.ReLU())
                
                stride_w = 1 if w == 1 else 2
                pool_w = 1 if w == 1 else 2
                layers.append(nn.MaxPool2d(kernel_size=(2, pool_w), stride=(2, stride_w)))
                in_channels = out_channels
            
            self.feature_extractor = nn.Sequential(*layers)
            
            # Adaptive pooling to strictly guarantee no dimension collapse
            self.adaptive_pool = nn.AdaptiveMaxPool2d((4, 1))
            self._to_linear = out_channels * 4 * 1
            
            self.classifier = nn.Sequential(
                nn.Flatten(),
                nn.Linear(self._to_linear, 64),
                nn.ReLU(),
                nn.Dropout(0.5),
                nn.Linear(64, 1),
                nn.Sigmoid()
            )

        def forward(self, x):
            # Enforce 4D tensor: [N, C, H, W]. If 3D [N, H, W] or [N, C, T], add a 1 at the end.
            if x.ndim == 3:
                if self.transpose_needed:
                    x = x.permute(0, 2, 1) # [N, T, C] -> [N, C, T]
                x = x.unsqueeze(-1)      # Add 1 at the end -> [N, C, T, 1] (making it strictly 4D)
            elif x.ndim == 2:
                x = x.unsqueeze(1).unsqueeze(-1) # [N, T] -> [N, 1, T, 1]
            elif x.ndim == 4:
                if self.transpose_needed and x.shape[1] > x.shape[2]:
                    x = x.permute(0, 3, 1, 2) # Adjust if needed
            
            x = self.feature_extractor(x)
            x = self.adaptive_pool(x)
            return self.classifier(x)

    sample_shape = X_sample.shape[1:] 
    model = DynamicGaitCNN(sample_shape, num_layers=num_layers)
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    
    return model

In [35]:
data_names = ['vector_magnitude','raw_axes','asymmetry_scalograms','cross_wavelet_coherence','phase_aligned_asymmetry','full_scalograms']

In [36]:
def get_features_by_name(i, alignment_method, feature_name):
    """
    Dispatcher function for non-scalogram features.
    """
    if feature_name == "vector_magnitude":
        return get_raw_vector_magnitude(i, alignment_method)
    elif feature_name == "raw_axes":
        return get_raw_vector_magnitude_with_axis(i, alignment_method)
    else:
        raise ValueError(f"Unknown non-scalogram feature name: {feature_name}")

In [37]:
def get_features_by_name_with_fold_adaptation(i, alignment_method, feature_name, train_scales, train_wavelet):
    """
    Dispatcher function for scalogram-based features, ensuring inner-fold scales 
    and wavelets are consistently applied.
    """
    # First, get the raw 12-channel axis segments required to compute scalograms
    raw_segments = get_raw_vector_magnitude_with_axis(i, alignment_method)
    if len(raw_segments) == 0:
        return np.empty((0,), dtype=np.float32)
        
    if feature_name == "asymmetry_scalograms":
        return compute_asymmetry_scalograms(raw_segments, scales=train_scales, wavelet=train_wavelet)
    elif feature_name == "cross_wavelet_coherence":
        return compute_cross_wavelet_coherence_scalograms(raw_segments, scales=train_scales, wavelet=train_wavelet)
    elif feature_name == "phase_aligned_asymmetry":
        return compute_phase_aligned_asymmetry_scalograms(raw_segments, scales=train_scales, wavelet=train_wavelet)
    elif feature_name == "full_scalograms":
        return compute_full_scalograms(raw_segments, scales=train_scales, wavelet=train_wavelet)
    else:
        raise ValueError(f"Unknown scalogram feature name: {feature_name}")

In [38]:
def get_raw_vector_magnitude(i, alignment_method):
    rec_acc, rec_gyro = prepare_patient_data(p_ids[i], condition_ids[i], alignment_method=alignment_method)
    
    patient_class = condition_ids[i].upper()
    overlap_mapping = {"PD": 0, "HC": 0, "DD": 0}
    overlap = overlap_mapping.get(patient_class, 0)
    hop = int(WINDOW_SIZE * (1 - overlap))
    
    all_tasks = set(rec_acc.keys()).union(set(rec_gyro.keys()))
    all_magnitudes = []
    
    for task_name in sorted(all_tasks):
        acc_data = rec_acc.get(task_name, None)
        gyro_data = rec_gyro.get(task_name, None)
        
        sample_modality = acc_data if acc_data is not None else gyro_data
        if sample_modality is None or "left" not in sample_modality:
            continue
            
        total_len = sample_modality["left"].shape[1]
        if total_len < WINDOW_SIZE:
            continue

        for start in range(0, total_len - WINDOW_SIZE + 1, hop):
            end = start + WINDOW_SIZE
            
            l_acc_seg = acc_data["left"][:, start:end].T if acc_data is not None else None
            r_acc_seg = acc_data["right"][:, start:end].T if acc_data is not None else None
            l_gyro_seg = gyro_data["left"][:, start:end].T if gyro_data is not None else None
            r_gyro_seg = gyro_data["right"][:, start:end].T if gyro_data is not None else None
            
            if l_acc_seg is None or r_acc_seg is None or l_gyro_seg is None or r_gyro_seg is None:
                continue
                
            mag_l_gyro = np.linalg.norm(l_gyro_seg, axis=-1)
            mag_r_gyro = np.linalg.norm(r_gyro_seg, axis=-1)
            mag_l_acc  = np.linalg.norm(l_acc_seg, axis=-1)
            mag_r_acc  = np.linalg.norm(r_acc_seg, axis=-1)
            
            processed_streams = []
            for stream in [mag_l_gyro, mag_r_gyro, mag_l_acc, mag_r_acc]:
                stream = np.array(stream).flatten()
                if len(stream) < WINDOW_SIZE:
                    stream = np.pad(stream, (0, WINDOW_SIZE - len(stream)), 'constant')
                elif len(stream) > WINDOW_SIZE:
                    stream = stream[:WINDOW_SIZE]
                processed_streams.append(stream)
            
            combined_streams = np.stack(processed_streams, axis=0)
            all_magnitudes.append(combined_streams)
            
    if not all_magnitudes:
        return np.empty((0, 4, WINDOW_SIZE), dtype=np.float32)
        
    return np.array(all_magnitudes, dtype=np.float32)

In [39]:
def get_raw_vector_magnitude_with_axis(i, alignment_method):
    rec_acc, rec_gyro = prepare_patient_data(p_ids[i], condition_ids[i], alignment_method=alignment_method)
    
    patient_class = condition_ids[i].upper()
    overlap_mapping = {"PD": 0, "HC": 0, "DD": 0}
    overlap = overlap_mapping.get(patient_class, 0)
    hop = int(WINDOW_SIZE * (1 - overlap))
    
    all_tasks = set(rec_acc.keys()).union(set(rec_gyro.keys()))
    all_combined_axes = []
    
    for task_name in sorted(all_tasks):
        acc_data = rec_acc.get(task_name, None)
        gyro_data = rec_gyro.get(task_name, None)
        
        sample_modality = acc_data if acc_data is not None else gyro_data
        if sample_modality is None or "left" not in sample_modality:
            continue
            
        total_len = sample_modality["left"].shape[1]
        if total_len < WINDOW_SIZE:
            continue

        for start in range(0, total_len - WINDOW_SIZE + 1, hop):
            end = start + WINDOW_SIZE
            
            l_acc_seg = acc_data["left"][:, start:end] if acc_data is not None else None
            r_acc_seg = acc_data["right"][:, start:end] if acc_data is not None else None
            l_gyro_seg = gyro_data["left"][:, start:end] if gyro_data is not None else None
            r_gyro_seg = gyro_data["right"][:, start:end] if gyro_data is not None else None
            
            if l_acc_seg is None or r_acc_seg is None or l_gyro_seg is None or r_gyro_seg is None:
                continue
                
            combined_streams = np.concatenate([l_gyro_seg, r_gyro_seg, l_acc_seg, r_acc_seg], axis=0)
            all_combined_axes.append(combined_streams)
            
    if not all_combined_axes:
        return np.empty((0, 12, WINDOW_SIZE), dtype=np.float32)
        
    return np.array(all_combined_axes, dtype=np.float32)
    

In [40]:
def compute_asymmetry_scalograms(data_n_12_t, scales=np.arange(1, 32), wavelet="morl", epsilon=1e-8):
    N, _, T = data_n_12_t.shape
    num_scales = len(scales)
    
    sl_gyro = data_n_12_t[:, 0:3, :]
    sr_gyro = data_n_12_t[:, 3:6, :]
    sl_accel = data_n_12_t[:, 6:9, :]
    sr_accel = data_n_12_t[:, 9:12, :]
    
    gyro_asymmetry = (sr_gyro - sl_gyro) / (sr_gyro + sl_gyro + epsilon)
    accel_asymmetry = (sr_accel - sl_accel) / (sr_accel + sl_accel + epsilon)
    
    cwt_asymmetry_tensor = np.empty((N, 6, num_scales, T), dtype=np.float32)
    
    for i in range(N):
        for c in range(3):
            signal = gyro_asymmetry[i, c, :]
            coeffs, _ = pywt.cwt(signal, scales, wavelet)
            cwt_asymmetry_tensor[i, c, :, :] = np.abs(coeffs)
            
    for i in range(N):
        for c in range(3):
            signal = accel_asymmetry[i, c, :]
            coeffs, _ = pywt.cwt(signal, scales, wavelet)
            cwt_asymmetry_tensor[i, c + 3, :, :] = np.abs(coeffs)
            
    return cwt_asymmetry_tensor

In [41]:
def compute_cross_wavelet_coherence_scalograms(data_n_12_t, scales=np.arange(1, 32), wavelet="morl", epsilon=1e-8):
    N, _, T = data_n_12_t.shape
    num_scales = len(scales)
    
    sl_gyro = data_n_12_t[:, 0:3, :]
    sr_gyro = data_n_12_t[:, 3:6, :]
    sl_accel = data_n_12_t[:, 6:9, :]
    sr_accel = data_n_12_t[:, 9:12, :]
    
    cwt_xwc_tensor = np.empty((N, 6, num_scales, T), dtype=np.float32)
    
    for i in range(N):
        for c in range(3):
            W_l, _ = pywt.cwt(sl_gyro[i, c, :], scales, wavelet)
            W_r, _ = pywt.cwt(sr_gyro[i, c, :], scales, wavelet)
            W_rl = W_r * np.conj(W_l)
            denom = np.sqrt(np.abs(W_l)**2 * np.abs(W_r)**2) + epsilon
            xwc = np.abs(W_rl) / denom
            cwt_xwc_tensor[i, c, :, :] = xwc
            
    for i in range(N):
        for c in range(3):
            W_l, _ = pywt.cwt(sl_accel[i, c, :], scales, wavelet)
            W_r, _ = pywt.cwt(sr_accel[i, c, :], scales, wavelet)
            W_rl = W_r * np.conj(W_l)
            denom = np.sqrt(np.abs(W_l)**2 * np.abs(W_r)**2) + epsilon
            xwc = np.abs(W_rl) / denom
            cwt_xwc_tensor[i, c + 3, :, :] = xwc
            
    return cwt_xwc_tensor

In [42]:
def compute_phase_aligned_asymmetry_scalograms(data_n_12_t, scales=np.arange(1, 32), wavelet="morl", epsilon=1e-8):
    N, _, T = data_n_12_t.shape
    num_scales = len(scales)
    
    sl_gyro = data_n_12_t[:, 0:3, :]
    sr_gyro = data_n_12_t[:, 3:6, :]
    sl_accel = data_n_12_t[:, 6:9, :]
    sr_accel = data_n_12_t[:, 9:12, :]
    
    gyro_asymmetry = np.empty((N, 3, T), dtype=np.float32)
    accel_asymmetry = np.empty((N, 3, T), dtype=np.float32)
    
    for i in range(N):
        for c in range(3):
            sig_l = sl_gyro[i, c, :]
            sig_r = sr_gyro[i, c, :]
            
            phase_l = np.angle(hilbert(sig_l))
            phase_r = np.angle(hilbert(sig_r))
            phase_diff = phase_r - phase_l
            
            analytic_r = hilbert(sig_r)
            sig_r_aligned = np.real(analytic_r * np.exp(-1j * phase_diff))
            
            gyro_asymmetry[i, c, :] = (sig_r_aligned - sig_l) / (sig_r_aligned + sig_l + epsilon)

    for i in range(N):
        for c in range(3):
            sig_l = sl_accel[i, c, :]
            sig_r = sr_accel[i, c, :]
            
            phase_l = np.angle(hilbert(sig_l))
            phase_r = np.angle(hilbert(sig_r))
            phase_diff = phase_r - phase_l
            
            analytic_r = hilbert(sig_r)
            sig_r_aligned = np.real(analytic_r * np.exp(-1j * phase_diff))
            
            accel_asymmetry[i, c, :] = (sig_r_aligned - sig_l) / (sig_r_aligned + sig_l + epsilon)
            
    cwt_asymmetry_tensor = np.empty((N, 6, num_scales, T), dtype=np.float32)
    
    for i in range(N):
        for c in range(3):
            signal = gyro_asymmetry[i, c, :]
            coeffs, _ = pywt.cwt(signal, scales, wavelet)
            cwt_asymmetry_tensor[i, c, :, :] = np.abs(coeffs)
            
    for i in range(N):
        for c in range(3):
            signal = accel_asymmetry[i, c, :]
            coeffs, _ = pywt.cwt(signal, scales, wavelet)
            cwt_asymmetry_tensor[i, c + 3, :, :] = np.abs(coeffs)
            
    return cwt_asymmetry_tensor

In [43]:
def compute_full_scalograms(data_n_12_t, scales=np.arange(1, 32), wavelet="morl"):
    N, C, T = data_n_12_t.shape
    num_scales = len(scales)
    
    cwt_tensor = np.empty((N, C, num_scales, T), dtype=np.float32)
    
    for i in range(N):
        for c in range(C):
            signal = data_n_12_t[i, c, :]
            coeffs, _ = pywt.cwt(signal, scales, wavelet)
            cwt_tensor[i, c, :, :] = np.abs(coeffs)
            
    return cwt_tensor

In [44]:
def _load_features_for_subjects(
    subject_list, 
    p_ids, 
    binary_labels, 
    alignment_method, 
    WINDOW_SIZE, 
    scales=None, 
    wavelet=None, 
    feature_name="vector_magnitude", 
    return_subjects=False
):
    X_list = []
    y_list = []
    window_subjects_list = []
    
    is_scalogram_feature = feature_name in [
        "asymmetry_scalograms", 
        "cross_wavelet_coherence", 
        "phase_aligned_asymmetry", 
        "full_scalograms"
    ]
    
    for subj in subject_list:
        indices = np.where(p_ids == subj)[0]
        if len(indices) == 0:
            continue
            
        subj_label = binary_labels[indices[0]]
        
        for idx in indices:
            if is_scalogram_feature:
                current_scales = scales if scales is not None else np.arange(1, 32)
                current_wavelet = wavelet if wavelet is not None else "morl"
                
                features = get_features_by_name_with_fold_adaptation(
                    i=idx,
                    alignment_method=alignment_method,
                    feature_name=feature_name,
                    train_scales=current_scales,
                    train_wavelet=current_wavelet
                )
            else:
                features = get_features_by_name(
                    i=idx,
                    alignment_method=alignment_method,
                    feature_name=feature_name
                )
            
            if features is not None and len(features) > 0:
                for window_feat in features:
                    X_list.append(window_feat)
                    y_list.append(subj_label)
                    window_subjects_list.append(subj)
                    
    if not X_list:
        if return_subjects:
            return np.array([]), np.array([]), np.array([])
        return np.array([]), np.array([])
        
    X = np.array(X_list, dtype=np.float32)
    y = np.array(y_list, dtype=np.float32)
    
    if return_subjects:
        window_subjects = np.array(window_subjects_list)
        return X, y, window_subjects
        
    return X, y

In [45]:
def balance_matrices_subject_wise(X_list_c0, X_list_c1, GLOBAL_SEED=42):
    """
    Identifies the majority class by total window count, calculates the 
    required budget to match the minority class, and samples an identical 
    number of windows across all subjects of the majority class.
    """
    c0_windows_per_sub = [sub.shape[0] for sub in X_list_c0]
    c1_windows_per_sub = [sub.shape[0] for sub in X_list_c1]
    
    total_c0 = sum(c0_windows_per_sub)
    total_c1 = sum(c1_windows_per_sub)
    
    if total_c0 == 0 or total_c1 == 0:
        return np.concatenate(X_list_c0, axis=0) if X_list_c0 else np.array([]), \
               np.concatenate(X_list_c1, axis=0) if X_list_c1 else np.array([])

    if total_c0 == total_c1:
        X_out_c0 = np.concatenate(X_list_c0, axis=0)
        X_out_c1 = np.concatenate(X_list_c1, axis=0)
        return X_out_c0, X_out_c1

    if total_c1 > total_c0:
        maj_list = X_list_c1
        maj_counts = np.array(c1_windows_per_sub)
        target_total = total_c0
        is_c1_majority = True
    else:
        maj_list = X_list_c0
        maj_counts = np.array(c0_windows_per_sub)
        target_total = total_c1
        is_c1_majority = False

    num_maj_subs = len(maj_list)
    allocations = np.zeros(num_maj_subs, dtype=int)
    remaining_target = target_total
    active_subs = np.ones(num_maj_subs, dtype=bool)

    while remaining_target > 0 and np.any(active_subs):
        num_active = np.sum(active_subs)
        base_share = remaining_target // num_active
        remainder = remaining_target % num_active
        
        if base_share == 0:
            chosen_indices = np.where(active_subs)[0][:remaining_target]
            for idx in chosen_indices:
                allocations[idx] += 1
            break
            
        for i in range(num_maj_subs):
            if active_subs[i]:
                share = base_share + (1 if remainder > 0 else 0)
                remainder -= 1 if remainder > 0 else 0
                
                available = maj_counts[i] - allocations[i]
                take = min(share, available)
                
                allocations[i] += take
                remaining_target -= take
                
                if allocations[i] == maj_counts[i]:
                    active_subs[i] = False

    processed_maj_list = []
    rng = np.random.default_rng(GLOBAL_SEED)
    for i, sub_windows in enumerate(maj_list):
        n_needed = allocations[i]
        if n_needed > 0:
            chosen_indices = rng.choice(sub_windows.shape[0], size=n_needed, replace=False)
            processed_maj_list.append(sub_windows[chosen_indices])
            
    X_processed_maj = np.concatenate(processed_maj_list, axis=0)

    if is_c1_majority:
        return np.concatenate(X_list_c0, axis=0), X_processed_maj
    else:
        return X_processed_maj, np.concatenate(X_list_c1, axis=0)

In [46]:
def _apply_patient_wise_undersampling(X, y, subject_list, window_subject_ids, global_seed):
    """Binds feature arrays into per-subject lists for class 0 (HC) and class 1 (PD) and balances them."""
    X_list_c0 = []
    X_list_c1 = []
    
    for subj in subject_list:
        subj_mask = (window_subject_ids == subj)
        subj_X = X[subj_mask]
        subj_y = y[subj_mask]
        
        if len(subj_X) == 0:
            continue
            
        if np.any(subj_y == 0):
            X_list_c0.append(subj_X[subj_y == 0])
        if np.any(subj_y == 1):
            X_list_c1.append(subj_X[subj_y == 1])
            
    if not X_list_c0 or not X_list_c1:
        return X, y
        
    X_bal_c0, X_bal_c1 = balance_matrices_subject_wise(X_list_c0, X_list_c1, GLOBAL_SEED=global_seed)
    
    X_balanced = np.concatenate([X_bal_c0, X_bal_c1], axis=0)
    y_balanced = np.concatenate([
        np.zeros(len(X_bal_c0), dtype=int),
        np.ones(len(X_bal_c1), dtype=int)
    ], axis=0)
    
    return X_balanced, y_balanced

In [47]:
def _fit_with_early_stopping(model, X_train, y_train, lr=0.001, batch_size=32, epochs=100, patience=10, verbose=1):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    
    criterion = nn.BCELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    # Split 10% of training data for internal early stopping validation
    val_split_idx = int(len(X_train) * 0.9)
    
    # Shuffle indices to ensure random 10% validation split
    indices = np.random.permutation(len(X_train))
    train_idx, val_idx = indices[:val_split_idx], indices[val_split_idx:]
    
    X_tr_sub, y_tr_sub = X_train[train_idx], y_train[train_idx]
    X_val_sub, y_val_sub = X_train[val_idx], y_train[val_idx]
    
    train_dataset = torch.utils.data.TensorDataset(torch.tensor(X_tr_sub, dtype=torch.float32), torch.tensor(y_tr_sub, dtype=torch.float32))
    train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    
    val_dataset = torch.utils.data.TensorDataset(torch.tensor(X_val_sub, dtype=torch.float32), torch.tensor(y_val_sub, dtype=torch.float32))
    val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    
    best_val_loss = float('inf')
    patience_counter = 0
    best_model_state = None
    
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        correct_train = 0
        total_train = 0
        
        for batch_x, batch_y in train_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            optimizer.zero_grad()
            
            outputs = model(batch_x).squeeze(-1)
            # Fix zero-dim or scalar mismatch issues if batch size is 1 or trailing squeezes drop dims
            if outputs.ndim == 0:
                outputs = outputs.unsqueeze(0)
                
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * batch_x.size(0)
            preds = (outputs >= 0.5).float()
            correct_train += (preds == batch_y).sum().item()
            total_train += batch_y.size(0)
            
        train_acc = correct_train / total_train if total_train > 0 else 0
        train_loss = running_loss / total_train if total_train > 0 else 0
        
        # Validation pass for early stopping
        model.eval()
        val_running_loss = 0.0
        correct_val = 0
        total_val = 0
        with torch.no_grad():
            for batch_x, batch_y in val_loader:
                batch_x, batch_y = batch_x.to(device), batch_y.to(device)
                outputs = model(batch_x).squeeze(-1)
                if outputs.ndim == 0:
                    outputs = outputs.unsqueeze(0)
                    
                loss = criterion(outputs, batch_y)
                
                val_running_loss += loss.item() * batch_x.size(0)
                preds = (outputs >= 0.5).float()
                correct_val += (preds == batch_y).sum().item()
                total_val += batch_y.size(0)
                
        val_loss = val_running_loss / total_val if total_val > 0 else float('inf')
        val_acc = correct_val / total_val if total_val > 0 else 0.0
        
        if verbose and (epoch + 1) % 5 == 0:
            print(f"        Epoch {epoch+1:03d}/{epochs:03d} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")
            
        # Early Stopping check
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            best_model_state = model.state_dict().copy()
        else:
            patience_counter += 1
            if patience_counter >= patience:
                if verbose:
                    print(f"        Early stopping triggered at epoch {epoch + 1}")
                break
                
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
        
    return model

In [48]:
def _ensure_4d_tensor(X):
    """Ensures input features are strictly 4-D format: (N, C, F, T)."""
    X = np.array(X)
    if X.ndim == 3:
        X = np.expand_dims(X, axis=2)
    return X


def _compute_ece(probs, preds, actuals, n_bins=10):
    bin_boundaries = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    n = len(probs)
    for i in range(n_bins):
        bin_lower, bin_upper = bin_boundaries[i], bin_boundaries[i+1]
        in_bin = np.logical_and(probs > bin_lower, probs <= bin_upper)
        bin_count = np.sum(in_bin)
        if bin_count > 0:
            bin_acc = np.mean(preds[in_bin] == actuals[in_bin])
            bin_conf = np.mean(probs[in_bin])
            ece += (bin_count / n) * np.abs(bin_acc - bin_conf)
    return float(ece)


In [49]:
import torch
import numpy as np
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score, 
    precision_recall_fscore_support, matthews_corrcoef, 
    roc_auc_score, average_precision_score, log_loss, brier_score_loss
)
from sklearn.linear_model import LogisticRegression

def _evaluate_at_subject_level(
    test_subjects,
    p_ids,
    binary_labels,
    model,
    alignment_method,
    WINDOW_SIZE,
    batch_size,
    scales=None,
    wavelet=None,
    feature_name="vector_magnitude"
):
    """
    Evaluates the trained model on test subjects by aggregating window-level 
    predictions into a single subject-level prediction, then computes comprehensive classification, 
    calibration, and uncertainty metrics.
    """
    model.eval()
    device = next(model.parameters()).device
    
    is_scalogram_feature = feature_name in [
        "asymmetry_scalograms", 
        "cross_wavelet_coherence", 
        "phase_aligned_asymmetry", 
        "full_scalograms"
    ]
    
    true_labels = []
    predicted_probabilities = []
    
    with torch.no_grad():
        for subj in test_subjects:
            indices = np.where(p_ids == subj)[0]
            if len(indices) == 0:
                continue
            
            subj_label = binary_labels[indices[0]]
            
            # Load features for the current test subject
            if is_scalogram_feature:
                X_subj, y_subj = _load_features_for_subjects(
                    [subj], p_ids, binary_labels, alignment_method, WINDOW_SIZE, 
                    scales=scales, wavelet=wavelet, feature_name=feature_name, return_subjects=False
                )
            else:
                X_subj, y_subj = _load_features_for_subjects(
                    [subj], p_ids, binary_labels, alignment_method, WINDOW_SIZE, 
                    feature_name=feature_name, return_subjects=False
                )
            
            if len(X_subj) == 0:
                continue
                
            # If the feature is 2D (e.g., [N, WINDOW_SIZE]), add a channel dimension to make it [N, WINDOW_SIZE, 1] 
            # so the 2D CNN feature extractor can safely treat the temporal axis and channel axis correctly 
            # without triggering size mismatch runtime errors during max pooling.
            if X_subj.ndim == 2:
                X_subj = np.expand_dims(X_subj, axis=-1)
                
            # Create DataLoader for subject windows
            dataset = torch.utils.data.TensorDataset(torch.tensor(X_subj, dtype=torch.float32))
            loader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=False)
            
            subj_probs = []
            for batch_x, in loader:
                # Ensure input tensor layout matches expected channels-first or channels-last depending on CNN definition.
                # If DynamicGaitCNN expects [B, C, L], permute from [B, L, C] if needed:
                if batch_x.ndim == 3 and batch_x.shape[-1] < batch_x.shape[1]:
                    batch_x = batch_x.permute(0, 2, 1)
                    
                batch_x = batch_x.to(device)
                outputs = model(batch_x).squeeze(-1)
                if outputs.ndim == 0:
                    outputs = outputs.unsqueeze(0)
                
                # Apply sigmoid if model outputs logits (assuming standard binary cross-entropy setup)
                probs = torch.sigmoid(outputs) if outputs.min() < 0 or outputs.max() > 1 else outputs
                subj_probs.extend(probs.cpu().numpy().tolist())
            
            if len(subj_probs) > 0:
                # Aggregate window probabilities (mean probability across windows for the subject)
                mean_subj_prob = np.mean(subj_probs)
                predicted_probabilities.append(mean_subj_prob)
                true_labels.append(subj_label)
                
    if len(true_labels) == 0:
        return None
        
    true_labels = np.array(true_labels)
    predicted_probabilities = np.array(predicted_probabilities)
    predicted_classes = (predicted_probabilities >= 0.5).astype(int)
    
    # Calculate evaluation metrics
    acc = accuracy_score(true_labels, predicted_classes)
    bal_acc = balanced_accuracy_score(true_labels, predicted_classes)
    macro_f1 = f1_score(true_labels, predicted_classes, average='macro', zero_division=0)
    precision, recall, _, _ = precision_recall_fscore_support(true_labels, predicted_classes, average=None, labels=[0, 1], zero_division=0)
    mcc = matthews_corrcoef(true_labels, predicted_classes)
    
    try:
        auroc = roc_auc_score(true_labels, predicted_probabilities)
    except ValueError:
        auroc = 0.5  # Fallback if only one class is present in the test batch
        
    try:
        auprc = average_precision_score(true_labels, predicted_probabilities)
    except ValueError:
        auprc = 0.0
        
    log_l = log_loss(true_labels, np.clip(predicted_probabilities, 1e-15, 1 - 1e-15))
    brier = brier_score_loss(true_labels, predicted_probabilities)
    
    calib_model = LogisticRegression()
    try:
        calib_model.fit(predicted_probabilities.reshape(-1, 1), true_labels)
        calib_slope = float(calib_model.coef_[0][0])
        calib_intercept = float(calib_model.intercept_[0])
    except:
        calib_slope, calib_intercept = 1.0, 0.0
        
    ece = _compute_ece(predicted_probabilities, predicted_classes, true_labels)
    
    confidences = np.where(predicted_classes == 1, predicted_probabilities, 1 - predicted_probabilities)
    conformal_coverage = float(np.mean(confidences >= 0.7))
    selective_risk = float(1.0 - accuracy_score(true_labels[confidences >= 0.7], predicted_classes[confidences >= 0.7])) if np.sum(confidences >= 0.7) > 0 else 0.0
    
    return {
        "accuracy": acc,
        "balanced_accuracy": bal_acc,
        "macro_f1": macro_f1,
        "precision_hc": precision[0],
        "precision_pd": precision[1],
        "recall_hc": recall[0],
        "recall_pd": recall[1],
        "mcc": mcc,
        "auroc": auroc,
        "auprc": auprc,
        "log_loss": log_l,
        "brier_score": brier,
        "calibration_slope": calib_slope,
        "calibration_intercept": calib_intercept,
        "ece": ece,
        "conformal_coverage": conformal_coverage,
        "selective_risk": selective_risk
    }

In [50]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score,
    precision_recall_fscore_support, matthews_corrcoef,
    roc_auc_score, average_precision_score, log_loss, brier_score_loss
)
from sklearn.linear_model import LogisticRegression

class LearnableWeightedFusion(nn.Module):
    """
    Combines CNN and RF outputs using standard softmax weights.
    Entropy regularization is applied during optimization to prevent collapse.
    """
    def __init__(self):
        super().__init__()
        self.w_logits = nn.Parameter(torch.tensor([0.0, 0.0], dtype=torch.float32))

    def forward(self, cnn_probs, rf_probs):
        weights = torch.softmax(self.w_logits, dim=0)
        fused_probs = weights[0] * cnn_probs + weights[1] * rf_probs
        return fused_probs, weights


def train_and_evaluate_ensemble_pipeline(
    X_tabular, y_tabular, p_ids, condition_ids, 
    positive_group='PD', 
    feature_name="vector_magnitude", WINDOW_SIZE=256
):
    """
    Trains and evaluates a late-fusion ensemble (CNN + Random Forest) using Leave-One-Subtype-Out (LOSO) 
    for DD subtypes against matched PD subjects.
    """
    p_ids = np.array(p_ids)
    condition_ids = np.array(condition_ids)
    
    # Identify all unique DD subtypes (everything except PD)
    dd_subtypes = sorted(list(set(condition_ids[condition_ids != positive_group])))
    print(f"Identified DD Subtypes for Leave-One-Out Evaluation: {dd_subtypes}")

    # Keep only records belonging to DD subtypes or PD
    valid_conditions = dd_subtypes + [positive_group]
    mask = np.isin(condition_ids, valid_conditions)
    p_ids = p_ids[mask]
    condition_ids = condition_ids[mask]
    X_tabular = np.array(X_tabular)[mask]
    
    # Binary labels: 0 for any DD subtype, 1 for PD
    binary_labels = np.array([1 if c == positive_group else 0 for c in condition_ids])
    
    unique_subjects = np.unique(p_ids)
    subject_labels = np.array([binary_labels[p_ids == s][0] for s in unique_subjects])

    n_inner_splits = 2
    n_seeds = 1

    alignment_space = ["task-marker"]
    cnn_layers_space = [2]
    lr_space = [0.001]
    batch_size_space = [16]
    
    is_scalogram_feature = feature_name in [
        "asymmetry_scalograms", 
        "cross_wavelet_coherence", 
        "phase_aligned_asymmetry", 
        "full_scalograms"
    ]
    
    if is_scalogram_feature:
        scales_space = [np.arange(1, 32)]
        wavelet_space = ["morl"]
    else:
        scales_space = [None]
        wavelet_space = [None]

    rf_param_grid = {
        "n_estimators": [50],
        "max_depth": [None],
        "min_samples_split": [2]
    }

    all_fold_metrics = []

    for seed in range(n_seeds):
        current_seed = 1000 + seed
        print(f"\n==========================================")
        print(f"    SEED / REPETITION {seed + 1} / {n_seeds} | LEAVE-ONE-SUBTYPE-OUT (DD vs {positive_group})")
        print(f"==========================================")
        
        # Get all PD subjects and shuffle them deterministically for balanced test sampling
        all_pd_subjects = np.unique(p_ids[condition_ids == positive_group])
        rng = np.random.default_rng(current_seed)
        rng.shuffle(all_pd_subjects)
        
        # Split PD subjects into chunks matching the number of DD subtypes to ensure fair evaluation across folds
        pd_sub_chunks = np.array_split(all_pd_subjects, len(dd_subtypes))

        for fold_idx, test_subtype in enumerate(dd_subtypes):
            print(f"\n------------------------------------------")
            print(f" [Outer Fold {fold_idx + 1}/{len(dd_subtypes)}] - Testing DD Subtype: '{test_subtype}'")
            print(f"------------------------------------------")
            
            # Test subjects: Current DD subtype + corresponding chunk of PD subjects
            test_dd_subs = np.unique(p_ids[condition_ids == test_subtype])
            test_pd_subs = pd_sub_chunks[fold_idx]
            outer_test_subjects = np.concatenate([test_dd_subs, test_pd_subs])
            
            # Train subjects: All other DD subtypes + all remaining PD subjects
            other_dd_subs = np.unique(p_ids[np.isin(condition_ids, dd_subtypes) & (condition_ids != test_subtype)])
            train_pd_subs = np.setdiff1d(all_pd_subjects, test_pd_subs)
            outer_train_subjects = np.concatenate([other_dd_subs, train_pd_subs])
            
            print(f"    -> Test DD Subtype '{test_subtype}' Count: {len(test_dd_subs)} subjects")
            print(f"    -> Test Matched PD Count: {len(test_pd_subs)} subjects")
            print(f"    -> Outer Train Subjects Count: {len(outer_train_subjects)}")

            # 1. CNN Hyperparameter Tuning (Inner CV)
            best_cnn_score = -1
            best_cnn_config = None
            inner_cv = StratifiedGroupKFold(n_splits=n_inner_splits, shuffle=True, random_state=100 + seed)
            inner_subject_labels = np.array([subject_labels[unique_subjects == s][0] for s in outer_train_subjects])
            
            for alignment in alignment_space:
                for num_layers in cnn_layers_space:
                    for lr in lr_space:
                        for bs in batch_size_space:
                            for scales in scales_space:
                                for wavelet in wavelet_space:
                                    
                                    inner_fold_accuracies = []
                                    for inner_fold, (inner_tr_idx, inner_val_idx) in enumerate(inner_cv.split(outer_train_subjects, inner_subject_labels, outer_train_subjects)):
                                        in_train_subs = outer_train_subjects[inner_tr_idx]
                                        in_val_subs = outer_train_subjects[inner_val_idx]
                                        
                                        if is_scalogram_feature:
                                            X_tr, y_tr, tr_window_subjs = _load_features_for_subjects(in_train_subs, p_ids, binary_labels, alignment, WINDOW_SIZE, scales=scales, wavelet=wavelet, feature_name=feature_name, return_subjects=True)
                                            X_val, y_val = _load_features_for_subjects(in_val_subs, p_ids, binary_labels, alignment, WINDOW_SIZE, scales=scales, wavelet=wavelet, feature_name=feature_name, return_subjects=False)
                                        else:
                                            X_tr, y_tr, tr_window_subjs = _load_features_for_subjects(in_train_subs, p_ids, binary_labels, alignment, WINDOW_SIZE, feature_name=feature_name, return_subjects=True)
                                            X_val, y_val = _load_features_for_subjects(in_val_subs, p_ids, binary_labels, alignment, WINDOW_SIZE, feature_name=feature_name, return_subjects=False)
                                        
                                        if len(X_tr) == 0 or len(X_val) == 0:
                                            continue
                                            
                                        if X_tr.ndim == 3:
                                            X_tr = np.expand_dims(X_tr, axis=-1)
                                        if X_val.ndim == 3:
                                            X_val = np.expand_dims(X_val, axis=-1)
                                        
                                        X_tr, y_tr = _apply_patient_wise_undersampling(X_tr, y_tr, in_train_subs, tr_window_subjs, global_seed=current_seed)
                                        
                                        model = build_gait_cnn_model(X_tr, num_layers=num_layers, learning_rate=lr, batch_size=bs)
                                        model = _fit_with_early_stopping(model, X_tr, y_tr, lr=lr, batch_size=bs, epochs=100, patience=10, verbose=0)
                                        
                                        model.eval()
                                        device = next(model.parameters()).device
                                        val_loader = torch.utils.data.DataLoader(
                                            torch.utils.data.TensorDataset(torch.tensor(X_val, dtype=torch.float32), torch.tensor(y_val, dtype=torch.float32)), 
                                            batch_size=bs, shuffle=False
                                        )
                                        
                                        correct_val, total_val = 0, 0
                                        with torch.no_grad():
                                            for batch_x, batch_y in val_loader:
                                                batch_x, batch_y = batch_x.to(device), batch_y.to(device)
                                                outputs = model(batch_x).squeeze(-1)
                                                if outputs.ndim == 0:
                                                    outputs = outputs.unsqueeze(0)
                                                preds = (outputs >= 0.5).float()
                                                correct_val += (preds == batch_y).sum().item()
                                                total_val += batch_y.size(0)
                                                
                                        val_acc = correct_val / total_val if total_val > 0 else 0.0
                                        inner_fold_accuracies.append(val_acc)
                                    
                                    if inner_fold_accuracies:
                                        mean_val_acc = np.mean(inner_fold_accuracies)
                                        if mean_val_acc > best_cnn_score:
                                            best_cnn_score = mean_val_acc
                                            best_cnn_config = {
                                                "alignment": alignment,
                                                "num_layers": num_layers,
                                                "lr": lr,
                                                "batch_size": bs,
                                                "scales": scales if is_scalogram_feature else None,
                                                "wavelet": wavelet if is_scalogram_feature else None
                                            }

            # 2. Random Forest Hyperparameter Tuning (Inner CV)
            best_rf_score = -1
            best_rf_config = None
            for n_est in rf_param_grid["n_estimators"]:
                for max_d in rf_param_grid["max_depth"]:
                    for min_split in rf_param_grid["min_samples_split"]:
                        inner_fold_accuracies = []
                        for inner_fold, (inner_tr_idx, inner_val_idx) in enumerate(inner_cv.split(outer_train_subjects, inner_subject_labels, outer_train_subjects)):
                            in_train_subs = outer_train_subjects[inner_tr_idx]
                            in_val_subs = outer_train_subjects[inner_val_idx]
                            
                            train_mask = np.isin(p_ids, in_train_subs)
                            val_mask = np.isin(p_ids, in_val_subs)
                            
                            X_tr_rf, y_tr_rf = X_tabular[train_mask], binary_labels[train_mask]
                            X_val_rf, y_val_rf = X_tabular[val_mask], binary_labels[val_mask]
                            
                            if len(X_tr_rf) == 0 or len(X_val_rf) == 0:
                                continue
                                
                            rf = RandomForestClassifier(n_estimators=n_est, max_depth=max_d, min_samples_split=min_split, random_state=current_seed, n_jobs=-1)
                            rf.fit(X_tr_rf, y_tr_rf)
                            val_preds = rf.predict(X_val_rf)
                            inner_fold_accuracies.append(accuracy_score(y_val_rf, val_preds))
                        
                        if inner_fold_accuracies:
                            mean_val_acc = np.mean(inner_fold_accuracies)
                            if mean_val_acc > best_rf_score:
                                best_rf_score = mean_val_acc
                                best_rf_config = {"n_estimators": n_est, "max_depth": max_d, "min_samples_split": min_split}

            # 3. Train Final Outer Models
            if is_scalogram_feature:
                X_outer_tr, y_outer_tr, outer_tr_window_subjs = _load_features_for_subjects(
                    outer_train_subjects, p_ids, binary_labels, best_cnn_config["alignment"], WINDOW_SIZE,
                    scales=best_cnn_config["scales"], wavelet=best_cnn_config["wavelet"], feature_name=feature_name, return_subjects=True
                )
            else:
                X_outer_tr, y_outer_tr, outer_tr_window_subjs = _load_features_for_subjects(
                    outer_train_subjects, p_ids, binary_labels, best_cnn_config["alignment"], WINDOW_SIZE,
                    feature_name=feature_name, return_subjects=True
                )
            
            if len(X_outer_tr) > 0:
                if X_outer_tr.ndim == 3:
                    X_outer_tr = np.expand_dims(X_outer_tr, axis=-1)
                X_outer_tr, y_outer_tr = _apply_patient_wise_undersampling(X_outer_tr, y_outer_tr, outer_train_subjects, outer_tr_window_subjs, global_seed=current_seed)
                
                final_cnn = build_gait_cnn_model(X_outer_tr, num_layers=best_cnn_config["num_layers"], learning_rate=best_cnn_config["lr"], batch_size=best_cnn_config["batch_size"])
                final_cnn = _fit_with_early_stopping(final_cnn, X_outer_tr, y_outer_tr, lr=best_cnn_config["lr"], batch_size=best_cnn_config["batch_size"], epochs=100, patience=10, verbose=0)

            outer_train_mask = np.isin(p_ids, outer_train_subjects)
            X_outer_tr_rf, y_outer_tr_rf = X_tabular[outer_train_mask], binary_labels[outer_train_mask]

            final_rf = RandomForestClassifier(
                n_estimators=best_rf_config["n_estimators"],
                max_depth=best_rf_config["max_depth"],
                min_samples_split=best_rf_config["min_samples_split"],
                random_state=current_seed,
                n_jobs=-1
            )
            final_rf.fit(X_outer_tr_rf, y_outer_tr_rf)

            # 4. Entropy-Regularized Fusion Optimization on Inner Validation Splits
            final_cnn.eval()
            cnn_device = next(final_cnn.parameters()).device
            
            cnn_inner_probs = []
            rf_inner_probs = []
            inner_val_labels = []
            
            for inner_fold, (inner_tr_idx, inner_val_idx) in enumerate(inner_cv.split(outer_train_subjects, inner_subject_labels, outer_train_subjects)):
                in_val_subs = outer_train_subjects[inner_val_idx]
                
                if is_scalogram_feature:
                    X_val_fold, y_val_fold, val_window_subjs = _load_features_for_subjects(in_val_subs, p_ids, binary_labels, best_cnn_config["alignment"], WINDOW_SIZE,
                        scales=best_cnn_config["scales"], wavelet=best_cnn_config["wavelet"], feature_name=feature_name, return_subjects=True)
                else:
                    X_val_fold, y_val_fold, val_window_subjs = _load_features_for_subjects(
                        in_val_subs, p_ids, binary_labels, best_cnn_config["alignment"], WINDOW_SIZE,
                        feature_name=feature_name, return_subjects=True
                    )
                
                if len(X_val_fold) == 0:
                    continue
                if X_val_fold.ndim == 3:
                    X_val_fold = np.expand_dims(X_val_fold, axis=-1)
                
                val_loader = torch.utils.data.DataLoader(
                    torch.utils.data.TensorDataset(torch.tensor(X_val_fold, dtype=torch.float32)), 
                    batch_size=best_cnn_config["batch_size"], shuffle=False
                )
                fold_cnn_preds = []
                with torch.no_grad():
                    for (bx,) in val_loader:
                        bx = bx.to(cnn_device)
                        out = final_cnn(bx).squeeze(-1)
                        if out.ndim == 0:
                            out = out.unsqueeze(0)
                        fold_cnn_preds.extend(out.cpu().numpy().tolist())
                fold_cnn_preds = np.array(fold_cnn_preds)
                
                for subj in in_val_subs:
                    subj_indices = np.where(p_ids == subj)[0]
                    if len(subj_indices) == 0:
                        continue
                    subj_label = binary_labels[subj_indices[0]]
                    
                    subj_window_mask = (val_window_subjs == subj)
                    if np.sum(subj_window_mask) > 0:
                        mean_c_prob = np.mean(fold_cnn_preds[subj_window_mask])
                    else:
                        mean_c_prob = 0.5

                    X_subj_rf = X_tabular[subj_indices]
                    rf_probs = final_rf.predict_proba(X_subj_rf)
                    if rf_probs.shape[1] > 1:
                        mean_r_prob = np.mean(rf_probs[:, 1])
                    else:
                        only_class = final_rf.classes_[0]
                        mean_r_prob = 1.0 if only_class == 1 else 0.0

                    cnn_inner_probs.append(mean_c_prob)
                    rf_inner_probs.append(mean_r_prob)
                    inner_val_labels.append(subj_label)

            fusion_layer = LearnableWeightedFusion().to(cnn_device)
            
            if len(inner_val_labels) > 0:
                t_cnn_probs = torch.tensor(cnn_inner_probs, dtype=torch.float32).to(cnn_device)
                t_rf_probs = torch.tensor(rf_inner_probs, dtype=torch.float32).to(cnn_device)
                t_labels = torch.tensor(inner_val_labels, dtype=torch.float32).to(cnn_device)
                
                optimizer = optim.Adam(fusion_layer.parameters(), lr=0.05)
                criterion = nn.BCELoss()
                
                fusion_layer.train()
                for epoch in range(100):
                    optimizer.zero_grad()
                    fused_preds, weights = fusion_layer(t_cnn_probs, t_rf_probs)
                    fused_preds = torch.clamp(fused_preds, 1e-7, 1.0 - 1e-7)
                    
                    bce_loss = criterion(fused_preds, t_labels)
                    entropy = -torch.sum(weights * torch.log(weights + 1e-8))
                    
                    lambda_entropy = 0.65
                    loss = bce_loss - (lambda_entropy * entropy)
                    
                    loss.backward()
                    optimizer.step()

            # 5. Evaluate on Locked Test Subjects
            true_labels = []
            ensemble_probabilities = []
            
            with torch.no_grad():
                if is_scalogram_feature:
                    X_test_all, _, test_window_subjs = _load_features_for_subjects(
                        outer_test_subjects, p_ids, binary_labels, best_cnn_config["alignment"], WINDOW_SIZE,
                        scales=best_cnn_config["scales"], wavelet=best_cnn_config["wavelet"], feature_name=feature_name, return_subjects=True
                    )
                else:
                    X_test_all, _, test_window_subjs = _load_features_for_subjects(
                        outer_test_subjects, p_ids, binary_labels, best_cnn_config["alignment"], WINDOW_SIZE,
                        feature_name=feature_name, return_subjects=True
                    )
                
                if len(X_test_all) > 0:
                    if X_test_all.ndim == 3:
                        X_test_all = np.expand_dims(X_test_all, axis=-1)
                    
                    test_loader = torch.utils.data.DataLoader(
                        torch.utils.data.TensorDataset(torch.tensor(X_test_all, dtype=torch.float32)), 
                        batch_size=best_cnn_config["batch_size"], shuffle=False
                    )
                    
                    all_test_cnn_preds = []
                    for (batch_x,) in test_loader:
                        batch_x = batch_x.to(cnn_device)
                        outputs = final_cnn(batch_x).squeeze(-1)
                        if outputs.ndim == 0:
                            outputs = outputs.unsqueeze(0)
                        all_test_cnn_preds.extend(outputs.cpu().numpy().tolist())
                    all_test_cnn_preds = np.array(all_test_cnn_preds)

                for subj in outer_test_subjects:
                    subj_indices = np.where(p_ids == subj)[0]
                    if len(subj_indices) == 0:
                        continue
                    
                    subj_label = binary_labels[subj_indices[0]]
                    
                    subj_window_mask = (test_window_subjs == subj) if len(X_test_all) > 0 else np.array([])
                    if len(subj_window_mask) > 0 and np.sum(subj_window_mask) > 0:
                        mean_cnn_prob = np.mean(all_test_cnn_preds[subj_window_mask])
                    else:
                        mean_cnn_prob = 0.5

                    X_subj_rf = X_tabular[subj_indices]
                    rf_probs = final_rf.predict_proba(X_subj_rf)
                    if rf_probs.shape[1] > 1:
                        mean_rf_prob = np.mean(rf_probs[:, 1])
                    else:
                        only_class = final_rf.classes_[0]
                        mean_rf_prob = 1.0 if only_class == 1 else 0.0

                    c_tensor = torch.tensor([mean_cnn_prob], dtype=torch.float32).to(cnn_device)
                    r_tensor = torch.tensor([mean_rf_prob], dtype=torch.float32).to(cnn_device)
                    fused_prob, _ = fusion_layer(c_tensor, r_tensor)
                    
                    true_labels.append(subj_label)
                    ensemble_probabilities.append(fused_prob.item())

            if len(true_labels) > 0:
                true_labels = np.array(true_labels)
                ensemble_probabilities = np.array(ensemble_probabilities)
                predicted_classes = (ensemble_probabilities >= 0.5).astype(int)
                
                acc = accuracy_score(true_labels, predicted_classes)
                bal_acc = balanced_accuracy_score(true_labels, predicted_classes)
                macro_f1 = f1_score(true_labels, predicted_classes, average='macro', zero_division=0)
                precision, recall, _, _ = precision_recall_fscore_support(true_labels, predicted_classes, average=None, labels=[0, 1], zero_division=0)
                mcc = matthews_corrcoef(true_labels, predicted_classes)
                
                try:
                    auroc = roc_auc_score(true_labels, ensemble_probabilities)
                except ValueError:
                    auroc = 0.5 
                    
                try:
                    auprc = average_precision_score(true_labels, ensemble_probabilities)
                except ValueError:
                    auprc = 0.0
                    
                log_l = log_loss(true_labels, np.clip(ensemble_probabilities, 1e-15, 1 - 1e-15))
                brier = brier_score_loss(true_labels, ensemble_probabilities)
                
                calib_model = LogisticRegression()
                try:
                    calib_model.fit(ensemble_probabilities.reshape(-1, 1), true_labels)
                    calib_slope = float(calib_model.coef_[0][0])
                    calib_intercept = float(calib_model.intercept_[0])
                except:
                    calib_slope, calib_intercept = 1.0, 0.0
                    
                confidences = np.where(predicted_classes == 1, ensemble_probabilities, 1 - ensemble_probabilities)
                conformal_coverage = float(np.mean(confidences >= 0.7))
                selective_risk = float(1.0 - accuracy_score(true_labels[confidences >= 0.7], predicted_classes[confidences >= 0.7])) if np.sum(confidences >= 0.7) > 0 else 0.0
                
                sub_metrics = {
                    "test_subtype": test_subtype,
                    "accuracy": acc,
                    "balanced_accuracy": bal_acc,
                    "macro_f1": macro_f1,
                    "precision_dd": precision[0],
                    "precision_pd": precision[1],
                    "recall_dd": recall[0],
                    "recall_pd": recall[1],
                    "mcc": mcc,
                    "auroc": auroc,
                    "auprc": auprc,
                    "log_loss": log_l,
                    "brier_score": brier,
                    "calibration_slope": calib_slope,
                    "calibration_intercept": calib_intercept,
                    "ece": 0.0,
                    "conformal_coverage": conformal_coverage,
                    "selective_risk": selective_risk
                }
                
                all_fold_metrics.append(sub_metrics)
                print(f"    -> Test Subtype '{test_subtype}' | Acc: {sub_metrics['accuracy']:.4f} | Balanced Acc: {sub_metrics['balanced_accuracy']:.4f} | AUROC: {sub_metrics['auroc']:.4f}")

    print(f"\n==========================================")
    print(f" LEAVE-ONE-SUBTYPE-OUT PIPELINE COMPLETED")
    print(f"==========================================")
    return all_fold_metrics

In [51]:
data_names = ['vector_magnitude','raw_axes','asymmetry_scalograms','cross_wavelet_coherence','phase_aligned_asymmetry','full_scalograms']

In [52]:
def print_aggregated_metrics(all_fold_metrics):
    """
    Prints aggregated metrics across all outer folds (Leave-One-Subtype-Out),
    displaying the metric value for each specific disease category tested.
    """
    if not all_fold_metrics:
        print("No fold metrics available to aggregate.")
        return

    # Identify all metric keys except 'test_subtype'
    metrics_keys = [k for k in all_fold_metrics[0].keys() if k != 'test_subtype']

    print("\n==================================================")
    print("      PER-SUBTYPE AND AGGREGATED METRICS")
    print("==================================================")

    # Print breakdown per disease category first
    for fold in all_fold_metrics:
        subtype = fold['test_subtype']
        print(f"\n--- Disease Category / Subtype: {subtype} ---")
        for key in metrics_keys:
            val = fold[key]
            print(f"  {key:<22}: {val:.4f}")

    print("\n==================================================")
    print("               OVERALL AVERAGE (MEAN ± STD)")
    print("==================================================")
    for key in metrics_keys:
        values = [float(fold[key]) for fold in all_fold_metrics]
        mean_val = np.mean(values)
        std_val = np.std(values)
        print(f"{key:<25}: {mean_val:.4f} ± {std_val:.4f}")
    print("==================================================\n")

In [53]:
data_names = ['vector_magnitude','raw_axes','asymmetry_scalograms','cross_wavelet_coherence','phase_aligned_asymmetry','full_scalograms']

In [54]:
k = 0
all_fold_metrics =train_and_evaluate_ensemble_pipeline(X_pd_dd, y_pd_dd, p_ids, condition_ids, positive_group='PD', feature_name=data_names[k], WINDOW_SIZE=256)
print_aggregated_metrics(all_fold_metrics)

Identified DD Subtypes for Leave-One-Out Evaluation: ['Essential Tremor', 'Multiple Sclerosis', 'Other Movement Disorders']

    SEED / REPETITION 1 / 1 | LEAVE-ONE-SUBTYPE-OUT (DD vs PD)

------------------------------------------
 [Outer Fold 1/3] - Testing DD Subtype: 'Essential Tremor'
------------------------------------------
    -> Test DD Subtype 'Essential Tremor' Count: 28 subjects
    -> Test Matched PD Count: 97 subjects
    -> Outer Train Subjects Count: 265
    -> Test Subtype 'Essential Tremor' | Acc: 0.8160 | Balanced Acc: 0.6782 | AUROC: 0.7666

------------------------------------------
 [Outer Fold 2/3] - Testing DD Subtype: 'Multiple Sclerosis'
------------------------------------------
    -> Test DD Subtype 'Multiple Sclerosis' Count: 11 subjects
    -> Test Matched PD Count: 97 subjects
    -> Outer Train Subjects Count: 282
    -> Test Subtype 'Multiple Sclerosis' | Acc: 0.8241 | Balanced Acc: 0.6200 | AUROC: 0.7001

------------------------------------------
 [

In [55]:
k = 1
all_fold_metrics =train_and_evaluate_ensemble_pipeline(X_pd_dd, y_pd_dd, p_ids, condition_ids, positive_group='PD', feature_name=data_names[k], WINDOW_SIZE=256)
print_aggregated_metrics(all_fold_metrics)

Identified DD Subtypes for Leave-One-Out Evaluation: ['Essential Tremor', 'Multiple Sclerosis', 'Other Movement Disorders']

    SEED / REPETITION 1 / 1 | LEAVE-ONE-SUBTYPE-OUT (DD vs PD)

------------------------------------------
 [Outer Fold 1/3] - Testing DD Subtype: 'Essential Tremor'
------------------------------------------
    -> Test DD Subtype 'Essential Tremor' Count: 28 subjects
    -> Test Matched PD Count: 97 subjects
    -> Outer Train Subjects Count: 265
    -> Test Subtype 'Essential Tremor' | Acc: 0.8160 | Balanced Acc: 0.6782 | AUROC: 0.7743

------------------------------------------
 [Outer Fold 2/3] - Testing DD Subtype: 'Multiple Sclerosis'
------------------------------------------
    -> Test DD Subtype 'Multiple Sclerosis' Count: 11 subjects
    -> Test Matched PD Count: 97 subjects
    -> Outer Train Subjects Count: 282
    -> Test Subtype 'Multiple Sclerosis' | Acc: 0.8426 | Balanced Acc: 0.5900 | AUROC: 0.7245

------------------------------------------
 [

In [56]:
k = 2
all_fold_metrics =train_and_evaluate_ensemble_pipeline(X_pd_dd, y_pd_dd, p_ids, condition_ids, positive_group='PD', feature_name=data_names[k], WINDOW_SIZE=256)
print_aggregated_metrics(all_fold_metrics)

Identified DD Subtypes for Leave-One-Out Evaluation: ['Essential Tremor', 'Multiple Sclerosis', 'Other Movement Disorders']

    SEED / REPETITION 1 / 1 | LEAVE-ONE-SUBTYPE-OUT (DD vs PD)

------------------------------------------
 [Outer Fold 1/3] - Testing DD Subtype: 'Essential Tremor'
------------------------------------------
    -> Test DD Subtype 'Essential Tremor' Count: 28 subjects
    -> Test Matched PD Count: 97 subjects
    -> Outer Train Subjects Count: 265
    -> Test Subtype 'Essential Tremor' | Acc: 0.8160 | Balanced Acc: 0.6782 | AUROC: 0.7640

------------------------------------------
 [Outer Fold 2/3] - Testing DD Subtype: 'Multiple Sclerosis'
------------------------------------------
    -> Test DD Subtype 'Multiple Sclerosis' Count: 11 subjects
    -> Test Matched PD Count: 97 subjects
    -> Outer Train Subjects Count: 282
    -> Test Subtype 'Multiple Sclerosis' | Acc: 0.8426 | Balanced Acc: 0.6303 | AUROC: 0.7366

------------------------------------------
 [

In [57]:
k = 3
all_fold_metrics =train_and_evaluate_ensemble_pipeline(X_pd_dd, y_pd_dd, p_ids, condition_ids, positive_group='PD', feature_name=data_names[k], WINDOW_SIZE=256)
print_aggregated_metrics(all_fold_metrics)

Identified DD Subtypes for Leave-One-Out Evaluation: ['Essential Tremor', 'Multiple Sclerosis', 'Other Movement Disorders']

    SEED / REPETITION 1 / 1 | LEAVE-ONE-SUBTYPE-OUT (DD vs PD)

------------------------------------------
 [Outer Fold 1/3] - Testing DD Subtype: 'Essential Tremor'
------------------------------------------
    -> Test DD Subtype 'Essential Tremor' Count: 28 subjects
    -> Test Matched PD Count: 97 subjects
    -> Outer Train Subjects Count: 265
    -> Test Subtype 'Essential Tremor' | Acc: 0.8400 | Balanced Acc: 0.7191 | AUROC: 0.7898

------------------------------------------
 [Outer Fold 2/3] - Testing DD Subtype: 'Multiple Sclerosis'
------------------------------------------
    -> Test DD Subtype 'Multiple Sclerosis' Count: 11 subjects
    -> Test Matched PD Count: 97 subjects
    -> Outer Train Subjects Count: 282
    -> Test Subtype 'Multiple Sclerosis' | Acc: 0.8519 | Balanced Acc: 0.5951 | AUROC: 0.7198

------------------------------------------
 [

In [58]:
k = 4
all_fold_metrics =train_and_evaluate_ensemble_pipeline(X_pd_dd, y_pd_dd, p_ids, condition_ids, positive_group='PD', feature_name=data_names[k], WINDOW_SIZE=256)
print_aggregated_metrics(all_fold_metrics)

Identified DD Subtypes for Leave-One-Out Evaluation: ['Essential Tremor', 'Multiple Sclerosis', 'Other Movement Disorders']

    SEED / REPETITION 1 / 1 | LEAVE-ONE-SUBTYPE-OUT (DD vs PD)

------------------------------------------
 [Outer Fold 1/3] - Testing DD Subtype: 'Essential Tremor'
------------------------------------------
    -> Test DD Subtype 'Essential Tremor' Count: 28 subjects
    -> Test Matched PD Count: 97 subjects
    -> Outer Train Subjects Count: 265
    -> Test Subtype 'Essential Tremor' | Acc: 0.8160 | Balanced Acc: 0.6782 | AUROC: 0.7857

------------------------------------------
 [Outer Fold 2/3] - Testing DD Subtype: 'Multiple Sclerosis'
------------------------------------------
    -> Test DD Subtype 'Multiple Sclerosis' Count: 11 subjects
    -> Test Matched PD Count: 97 subjects
    -> Outer Train Subjects Count: 282
    -> Test Subtype 'Multiple Sclerosis' | Acc: 0.7778 | Balanced Acc: 0.5942 | AUROC: 0.7020

------------------------------------------
 [

In [59]:
k = 5
all_fold_metrics =train_and_evaluate_ensemble_pipeline(X_pd_dd, y_pd_dd, p_ids, condition_ids, positive_group='PD', feature_name=data_names[k], WINDOW_SIZE=256)
print_aggregated_metrics(all_fold_metrics)

Identified DD Subtypes for Leave-One-Out Evaluation: ['Essential Tremor', 'Multiple Sclerosis', 'Other Movement Disorders']

    SEED / REPETITION 1 / 1 | LEAVE-ONE-SUBTYPE-OUT (DD vs PD)

------------------------------------------
 [Outer Fold 1/3] - Testing DD Subtype: 'Essential Tremor'
------------------------------------------
    -> Test DD Subtype 'Essential Tremor' Count: 28 subjects
    -> Test Matched PD Count: 97 subjects
    -> Outer Train Subjects Count: 265
    -> Test Subtype 'Essential Tremor' | Acc: 0.8320 | Balanced Acc: 0.7012 | AUROC: 0.7640

------------------------------------------
 [Outer Fold 2/3] - Testing DD Subtype: 'Multiple Sclerosis'
------------------------------------------
    -> Test DD Subtype 'Multiple Sclerosis' Count: 11 subjects
    -> Test Matched PD Count: 97 subjects
    -> Outer Train Subjects Count: 282
    -> Test Subtype 'Multiple Sclerosis' | Acc: 0.8426 | Balanced Acc: 0.5900 | AUROC: 0.7329

------------------------------------------
 [